In [1]:
!pip install ultralytics
!pip install roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 361.2/361.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 66.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2


In [4]:
from roboflow import Roboflow
import shutil
import os

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

In [6]:
rf = Roboflow(api_key=user_secrets.get_secret("ROBOFLOW_API_KEY"))

In [7]:
project = rf.workspace("workspace-5ujvu").project("basketball-players-fy4c2-vfsuv")
version = project.version(17)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Basketball-Players-17 in yolov11:: 100%|██████████| 652/652 [00:00<00:00, 8311.54it/s]


In [8]:
import os

os.listdir("/kaggle/working/Basketball-Players-17")

['valid',
 'README.dataset.txt',
 'data.yaml',
 'test',
 'README.roboflow.txt',
 'train']

In [9]:
for split in ["train", "valid", "test"]:
    print(f"\n{split}:")
    print(os.listdir(f"/kaggle/working/Basketball-Players-17/{split}"))


train:
['labels', 'images']

valid:
['labels', 'images']

test:
['labels', 'images']


In [10]:
print(open("/kaggle/working/Basketball-Players-17/data.yaml").read())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 7
names: ['Ball', 'Clock', 'Hoop', 'Overlay', 'Player', 'Ref', 'Scoreboard']

roboflow:
  workspace: workspace-5ujvu
  project: basketball-players-fy4c2-vfsuv
  version: 17
  license: CC BY 4.0
  url: https://universe.roboflow.com/workspace-5ujvu/basketball-players-fy4c2-vfsuv/dataset/17


In [11]:
from collections import Counter
import glob

label_files = glob.glob(
    "/kaggle/working/Basketball-Players-17/train/labels/*.txt"
)

class_counts = Counter()

for file in label_files:
    with open(file) as f:
        for line in f:
            if line.strip():
                class_id = int(line.split()[0])
                class_counts[class_id] += 1

print(class_counts)

Counter({4: 2021, 5: 495, 3: 294, 1: 249, 0: 213, 6: 211, 2: 203})


In [12]:
for split in ["train", "valid", "test"]:
    images = glob.glob(
        f"/kaggle/working/Basketball-Players-17/{split}/images/*"
    )
    print(split, len(images))

train 256
valid 32
test 32


In [13]:

src = "/kaggle/working/Basketball-Players-17"
dst = "/kaggle/working/Basketball-Players-PlayerOnly"

for split in ["train", "valid", "test"]:
    os.makedirs(f"{dst}/{split}/images", exist_ok=True)
    os.makedirs(f"{dst}/{split}/labels", exist_ok=True)

    image_dir = f"{src}/{split}/images"
    label_dir = f"{src}/{split}/labels"

    for label_file in os.listdir(label_dir):

        if not label_file.endswith(".txt"):
            continue

        src_label = os.path.join(label_dir, label_file)

        player_lines = []

        with open(src_label, "r") as f:
            for line in f:
                parts = line.strip().split()

                if not parts:
                    continue

                class_id = int(parts[0])

                # Player = class 4
                if class_id == 4:
                    parts[0] = "0"
                    player_lines.append(" ".join(parts))

        # Only keep images containing players
        if player_lines:
            # Copy image
            image_name = os.path.splitext(label_file)[0]

            for ext in [".jpg", ".jpeg", ".png"]:
                image_path = os.path.join(image_dir, image_name + ext)

                if os.path.exists(image_path):
                    shutil.copy(
                        image_path,
                        f"{dst}/{split}/images/{image_name}{ext}"
                    )
                    break

            # Write new label
            with open(
                f"{dst}/{split}/labels/{label_file}", "w"
            ) as f:
                f.write("\n".join(player_lines))

In [14]:
yaml_content = """
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1
names: ['Player']
"""

with open(f"{dst}/data.yaml", "w") as f:
    f.write(yaml_content)

In [15]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [16]:
model = YOLO("yolo11m.pt")

In [18]:
results = model.train(
    task="detect",
    mode="train",
    data="/kaggle/working/Basketball-Players-PlayerOnly/data.yaml",
    epochs=100,
    imgsz=640,
    plots=True,
    batch=16,
    patience=20
)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Basketball-Players-PlayerOnly/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=No

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      7.99G      1.798      2.056      1.543        124        640: 100% ━━━━━━━━━━━━ 16/16 2.8it/s 5.6s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.7it/s 0.6s
                   all         32        259    0.00917       0.34    0.00417    0.00148

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      8.06G      1.913      1.793      1.596        158        640: 100% ━━━━━━━━━━━━ 16/16 2.9it/s 5.6s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.0it/s 0.5s
                   all         32        259          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      8.06G       1.98      1.722      1.692        154        640: 100% ━━━━━━━━━━━━ 16/16 2.9it/s 5.6s0.3s
                 Class     Images  Instances      Box(

In [19]:
metrics = model.val(
    data="/kaggle/working/Basketball-Players-PlayerOnly/data.yaml",
    split="test",
    plots=True
)

print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1565.1±401.4 MB/s, size: 66.4 KB)
val: Scanning /kaggle/working/Basketball-Players-PlayerOnly/test/labels... 32 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 32/32 1.1Kit/s 0.0s
val: New cache created: /kaggle/working/Basketball-Players-PlayerOnly/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.2it/s 1.6s2.8s
                   all         32        260      0.947      0.886      0.957      0.717
Speed: 4.2ms preprocess, 26.1ms inference, 0.0ms loss, 4.5ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
mAP@0.5: 0.9566402779418615
mAP@0.5:0.95: 0.7171824299639835
Precision: 0.9466047421523947
Recall: 0.8864166780176865


In [20]:
results = model.predict(
    source="/kaggle/working/Basketball-Players-PlayerOnly/test/images",
    conf=0.25,
    save=True
)


image 1/32 /kaggle/working/Basketball-Players-PlayerOnly/test/images/Iowa_Nebraska-2023-03-06-college-men_mp4-17_jpg.rf.a15a766ff7f17bd4170ce424061b7322.jpg: 640x640 9 Players, 35.9ms
image 2/32 /kaggle/working/Basketball-Players-PlayerOnly/test/images/youtube-104_jpg.rf.704d110fa1e4a13daa5597d8fabbb915.jpg: 640x640 10 Players, 36.0ms
image 3/32 /kaggle/working/Basketball-Players-PlayerOnly/test/images/youtube-117_jpg.rf.d265a2d4fb60f7b9cde5864302643cc6.jpg: 640x640 6 Players, 35.9ms
image 4/32 /kaggle/working/Basketball-Players-PlayerOnly/test/images/youtube-11_jpg.rf.c4272cdc197ac9692ef31e73d0d7c237.jpg: 640x640 10 Players, 24.5ms
image 5/32 /kaggle/working/Basketball-Players-PlayerOnly/test/images/youtube-128_jpg.rf.b4b49202146a9975db6f0c6db98746e4.jpg: 640x640 13 Players, 23.0ms
image 6/32 /kaggle/working/Basketball-Players-PlayerOnly/test/images/youtube-131_jpg.rf.9ffab0eb3094bf459315ccd504596e3c.jpg: 640x640 1 Player, 23.0ms
image 7/32 /kaggle/working/Basketball-Players-PlayerOn

In [21]:
import os

for root, dirs, files in os.walk("/kaggle/working/runs"):
    for file in files:
        if file.endswith(".pt"):
            print(os.path.join(root, file))

/kaggle/working/runs/detect/train/weights/best.pt
/kaggle/working/runs/detect/train/weights/last.pt
